# Prerequisite

In [21]:
from pydantic import BaseModel
import json
import pandas as pd
from pydantic import ValidationError
from pandas import DataFrame
from ollama import generate
from transformers import AutoTokenizer, pipeline
from dotenv import load_dotenv
import os
import ast
import json
from huggingface_hub import login


In [22]:
class MCQQuestion(BaseModel):
    question1: str
    option_a1: str
    option_b1: str
    option_c1: str
    option_d1: str
    correct_option1: str
    question2: str
    option_a2: str
    option_b2: str
    option_c2: str
    option_d2: str
    correct_option2: str


In [23]:
def validate_mcq(mcq_json):
    try:
        return MCQQuestion.model_validate_json(mcq_json)
    except ValidationError as e:
        print(f"Validation failed: {e}")
        return None
        


def flatten_and_export_mcq(df: DataFrame, export_filename: str, mcq_column_name: str):
    ids = [x for val in df["id"] for x in (val, val+'-') ]
    result_df = pd.DataFrame({"id": ids})
    
    result_df['question'] = pd.concat([df[mcq_column_name].apply(lambda x: x.question1 if x else ""), df[mcq_column_name].apply(lambda x: x.question2 if x else "")], ignore_index=True)
    result_df['option_a'] = pd.concat([df[mcq_column_name].apply(lambda x: x.option_a1 if x else ""), df[mcq_column_name].apply(lambda x: x.option_a2 if x else "")], ignore_index=True)
    result_df['option_b'] = pd.concat([df[mcq_column_name].apply(lambda x: x.option_b1 if x else ""), df[mcq_column_name].apply(lambda x: x.option_b2 if x else "")], ignore_index=True) 
    result_df['option_c'] = pd.concat([df[mcq_column_name].apply(lambda x: x.option_c1 if x else ""), df[mcq_column_name].apply(lambda x: x.option_c2 if x else "")], ignore_index=True)
    result_df['option_d'] = pd.concat([df[mcq_column_name].apply(lambda x: x.option_d1 if x else ""), df[mcq_column_name].apply(lambda x: x.option_d2 if x else "")], ignore_index=True)
    result_df['correct_option'] = pd.concat([df[mcq_column_name].apply(lambda x: x.correct_option1 if x else ""),df[mcq_column_name].apply(lambda x: x.correct_option2 if x else "")],ignore_index=True)
    
    result_df.to_csv(export_filename, index=False)

In [24]:
def extract_json(text):
    start = text.find("{")
    end = text.find("}")
    text = text[start:end+1]
    return json.dumps(ast.literal_eval(text), ensure_ascii=False)

In [25]:

def generate_mcq(content, model_name, temperature):
    prompt = f"""
        À partir du contenu éducatif suivant, générez deux questions à choix multiple avec quatre options de réponse dont une seule est correcte.
        La question doit évaluer la compréhension des idées principales, et les options doivent être claires, informatives et pertinentes.
        Assurez-vous que les distracteurs (options incorrectes) suivent une interprétation logique mais incorrecte, basée sur des idées reçues ou des incompréhensions courantes du sujet.
        Les options de réponse doivent être aussi courtes que possible.

        IMPORTANT — FORMAT ABSOLU POUR LES CHAMPS 'correct_option1' ET 'correct_option2' :
        - Ces champs doivent contenir exactement **une seule lettre minuscule** parmi : a, b, c ou d.
        - **Exemples valides** : "a", "b", "c", "d".
        - **Interdits** : "a)", "A", "a.", "a )", "le texte de la réponse correcte", 1, true, etc.
        - La sortie JSON doit conserver ces champs comme chaînes (`"correct_option1": "a"`).

        Fournissez la sortie strictement au format JSON correspondant au schéma demandé (ne pas produire de texte hors-du-JSON).
        **Contenu éducatif :**
        {content}
    """
    
    generate_params = {
        'model': model_name,
        'options': {'temperature': temperature, 'num_ctx': 8192, 'top_p': 1}, 
        'prompt': prompt,
        'format': MCQQuestion.model_json_schema()
    }
    
    # Get a response
    response = generate(**generate_params)
    return response['response']

In [26]:
def generate_mcq_hf(content, model_name,tokenizer, temperature):
    prompt = f"""
        À partir du contenu éducatif suivant, générez exactement deux questions à choix multiple avec quatre options de réponse chacune (a, b, c, d), dont une seule est correcte.

        OBJECTIFS :
        - Les questions doivent évaluer la compréhension des idées principales.
        - Les distracteurs doivent être plausibles mais incorrects.
        - Les options doivent être courtes.
        - Les deux questions doivent être fournies dans un seul et unique objet JSON.
        - Aucun texte hors JSON n’est autorisé.

        CONTRAINTES STRICTES DE SORTIE :
        1. La sortie doit être STRICTEMENT un unique objet JSON valide.
        2. Interdiction ABSOLUE d’ajouter :
        - des blocs ```json
        - plusieurs objets JSON
        - du texte avant ou après le JSON
        - des explications ou commentaires
        3. Les champs "correct_option1" et "correct_option2" doivent contenir EXACTEMENT une lettre minuscule parmi : "a", "b", "c", "d".
        4. il faut utiliser des doubles quotes : "..." et NON '...'
        5. Le JSON doit contenir EXACTEMENT les 12 champs suivants :

        {{
            "question1": "...",
            "option_a1": "...",
            "option_b1": "...",
            "option_c1": "...",
            "option_d1": "...",
            "correct_option1": "a",
            "question2": "...",
            "option_a2": "...",
            "option_b2": "...",
            "option_c2": "...",
            "option_d2": "...",
            "correct_option2": "c"
        }}

        CONTENU ÉDUCATIF :
            {content}

            INSTRUCTION FINALE :
            Répondez UNIQUEMENT avec un unique objet JSON valide, sans aucun texte en dehors.
        """

    
    pipe = pipeline(
        "text-generation",
        model=model_name,
        tokenizer=tokenizer,
        device_map="cuda",
        dtype="bfloat16"
    )
    
    messages = [{"role": "user", "content": prompt}]
    
    response = pipe(
        messages,
        max_new_tokens=2048,
        temperature=temperature,
        top_p=1.0,
        do_sample=True,
        return_full_text=False
    )

    
    return extract_json(response[0]['generated_text'])

In [27]:
def get_checkpoint():
    try:
        with open("../data/checkpoints/start", "r") as start:
            start = start.readline()
            df_in_construction = pd.read_csv("../data/checkpoints/df_in_construction.csv")
    except FileNotFoundError:
        df_in_construction = pd.DataFrame()
        start = 0
    return int(start), df_in_construction

def save_checkpoint(start, df_in_construction):
    with open("../data/checkpoints/start", "w") as fic:
        fic.write(str(start))
    df_in_construction.to_csv("../data/checkpoints/df_in_construction.csv", index=False)

In [34]:
def for_a_model(df_test, model_name, save_name, use_ollama=False):
    if not use_ollama:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            use_fast=True,
            trust_remote_code=True
        )
    else:
        tokenizer = None
    
    start, df_in_construction = get_checkpoint()
    pas = 400
    
    for idx in range(start, len(df_test)):
        content = df_test.loc[idx, "content_raw"]
        nb_try = 0
        while True:
            try:
                generated = (
                    generate_mcq_hf(content, model_name, tokenizer, temperature=0.1)
                    if not use_ollama
                    else generate_mcq(content, model_name, temperature=0.1)
                )
                break
            except SyntaxError:
                print("SyntaxError détectée, relance...")
                nb_try += 1
                if nb_try == 5:
                    print("Nombre d'essai depassé, passage au Lisa Sheet suivant")
                    break
        
        df_in_construction.loc[idx, f"generated_{save_name}"] = generated
        
        if idx % pas == 0:
            save_checkpoint(idx, df_in_construction)
    
    df_test[save_name] = df_in_construction[f'generated_{save_name}'].apply(validate_mcq)
    flatten_and_export_mcq(df_test, f'../data/base_models/instruct/{save_name}.csv', save_name)
    
    # Clean for other model
    os.remove("../data/checkpoints/df_in_construction.csv")
    os.remove("../data/checkpoints/start")

In [29]:
load_dotenv()                  
HF_TOKEN = os.getenv("HF_TOKEN")  
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [30]:
df = pd.read_csv("../data/lisa_sheets.csv")

In [31]:
file_path = "../data/train_test_split/test_folders.json"

In [32]:
with open(file_path, "r", encoding="utf-8") as file:
    test_folders = json.load(file)

In [33]:
df_test = df[df.folder.isin(test_folders)].reset_index(drop=True)
print("Number of lisa sheets :", len(df_test))

Number of lisa sheets : 1592


# Instructed models

In [ ]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"
save_name = "llama3_1_8b"
for_a_model(df_test,model_name,save_name)

In [ ]:
model_name = "google/medgemma-4b-it"
save_name = "medGemma_4b"
for_a_model(df_test,model_name,save_name)

In [ ]:
model_name = "google/gemma-2-9b-it"
save_name = "gemma2_9b"
for_a_model(df_test,model_name,save_name)

In [ ]:
model_name = "google/medgemma-27b-it"
save_name = "medGemma_27b"

for_a_model(df_test,model_name,save_name)

In [19]:
model_name="hf.co/mradermacher/Llama3-Instruct-OpenBioLLM-8B-merged-i1-GGUF:latest"
save_name = "openbiollm_8b"
for_a_model(df_test,model_name,save_name, True)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

# Additional models

In [ ]:
model_name = "Qwen/Qwen3-0.6B"
save_name = "qwen3_6b"
for_a_model(df_test,model_name,save_name)

In [15]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
save_name = "mistral_7b"
for_a_model(df_test,model_name,save_name)

Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.32it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.32it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.32it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.32it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|████████████████████████████████████████████████████

SyntaxError détectée, relance...


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.32it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.33it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.32it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.33it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|████████████████████████████████████████████████████

SyntaxError détectée, relance...


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.33it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.33it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.33it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.33it/s]
Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Loading checkpoint shards: 100%|████████████████████████████████████████████████████

Validation failed: 2 validation errors for MCQQuestion
option_d1
  Field required [type=missing, input_value={'question1': "Quel est l... 'correct_option2': 'a'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_d2
  Field required [type=missing, input_value={'question1': "Quel est l... 'correct_option2': 'a'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
Validation failed: 2 validation errors for MCQQuestion
option_d1
  Field required [type=missing, input_value={'question1': "Quels exam... 'correct_option2': 'c'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_d2
  Field required [type=missing, input_value={'question1': "Quels exam... 'correct_option2': 'c'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


In [35]:
model_name = "utter-project/EuroLLM-9B-Instruct"
save_name = "eurollm_9b"
for_a_model(df_test,model_name,save_name)

Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|████████████████████████████████████████████

SyntaxError détectée, relance...


Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda


SyntaxError détectée, relance...


Loading checkpoint shards: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda


...

SyntaxError détectée, relance...


Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda


SyntaxError détectée, relance...


Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.53it/s]
Device set to use cuda
Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.53it/s]
Device set to use cuda
Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.54it/s]
Device set to use cuda
Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:02<00:00,  1.53it/s]
Device 

Validation failed: 6 validation errors for MCQQuestion
question2
  Field required [type=missing, input_value={'question1': 'Quelle est... 'correct_option1': 'a'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_a2
  Field required [type=missing, input_value={'question1': 'Quelle est... 'correct_option1': 'a'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_b2
  Field required [type=missing, input_value={'question1': 'Quelle est... 'correct_option1': 'a'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_c2
  Field required [type=missing, input_value={'question1': 'Quelle est... 'correct_option1': 'a'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_d2
  Field required [type=missing, input_value={'question1': 'Quelle est... 'correct_option1': 'a'}, input_type=dict]
    For furt

In [34]:
model_name = "swiss-ai/Apertus-8B-Instruct-2509"
save_name = "apertus_8B"
for_a_model(df_test,model_name,save_name)

0


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.17s/it]
Device set to use cuda


1


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.17s/it]
Device set to use cuda


2


...

1590


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.16s/it]
Device set to use cuda


1591


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.15s/it]
Device set to use cuda


Validation failed: 6 validation errors for MCQQuestion
question2
  Field required [type=missing, input_value={'question1': "Les facteu... 'correct_option1': 'd'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_a2
  Field required [type=missing, input_value={'question1': "Les facteu... 'correct_option1': 'd'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_b2
  Field required [type=missing, input_value={'question1': "Les facteu... 'correct_option1': 'd'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_c2
  Field required [type=missing, input_value={'question1': "Les facteu... 'correct_option1': 'd'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
option_d2
  Field required [type=missing, input_value={'question1': "Les facteu... 'correct_option1': 'd'}, input_type=dict]
    For furt